# CATATAN ADAPTASI UNTUK KAGGLE

Notebook ini awalnya dibuat untuk **Google Colab** (memakai `google.colab.drive` dan folder
`MyDrive/NEW/...`). Sekarang sudah disesuaikan untuk **Kaggle Notebook**, dengan dataset
bernama **`new-dataset`** yang isinya SEMUA file (xlsx & pkl) langsung di root dataset —
**benar-benar flat, tanpa subfolder apapun** (tidak ada `preprocessed_data`, tidak ada
`processed-data`, tidak ada `experiment_states`, dst).

## Langkah yang perlu kamu lakukan di Kaggle (sebelum run):

1. **Dataset `new-dataset` sudah kamu buat di Kaggle** berisi semua file (xlsx & pkl) langsung
   di root dataset (tidak ada subfolder sama sekali).
2. **Tambahkan dataset ke notebook** — klik **+ Add Input** (panel kanan) -> cari `new-dataset`
   -> **Add**, lalu restart session.
3. **Aktifkan GPU** (Settings -> Accelerator -> GPU) karena training dilakukan untuk 3 nilai
   dropout (0.2, 0.5, 0.8), masing-masing dengan 5-fold K-Fold.

## Perbaikan di versi ini:
- Semua pembacaan file dari dataset (`dataset_hasil_labelling_aspek.xlsx`, `aspect_le.pkl`,
  `sentiment_le.pkl`) sekarang lewat helper `find_input_file(...)` yang mencari file
  **langsung di root `INPUT_DIR`**, dan otomatis mencari satu tingkat ke bawah kalau ternyata
  Kaggle tetap membungkusnya dalam folder tambahan. Kalau file benar-benar tidak ada, cell akan
  menampilkan **isi folder dataset yang sebenarnya** supaya gampang di-debug — bukan cuma error
  generik.
- `INPUT_DIR` dideteksi otomatis dari `/kaggle/input` (bukan hardcode `new-dataset` saja), jadi
  kalau slug dataset kamu sedikit berbeda, notebook tetap jalan dan akan memberi peringatan.
- `drive.mount(...)` dihapus total (Kaggle tidak pakai Google Drive).
- Semua path pakai dua variabel utama:
  - `INPUT_DIR` = dataset Kaggle (read-only, FLAT, tanpa subfolder).
  - `WORK_DIR` = `/kaggle/working` (semua model, pickle, dataset olahan disimpan di sini;
    otomatis ikut tersimpan saat kamu klik **Save Version**).
- `BASE_DRIVE_PATH`, `SAVE_ROOT_PATH` tetap dipakai di cell-cell berikutnya (supaya struktur kode
  asli tidak perlu diubah total), tapi sekarang jadi subfolder di dalam `WORK_DIR` (writable).
- **Penting**: `/kaggle/input` bersifat *read-only* — semua hasil (model .keras, tokenizer.pkl,
  dataset olahan) disimpan ke `/kaggle/working`.

## Uji coba parameter Dropout (0.2 / 0.5 / 0.8)
- `DROPOUT_VALUES = [0.2, 0.5, 0.8]` di cell konfigurasi paling atas.
- Cell training **klasifikasi aspek** dan **sentimen per aspek** melatih model untuk
  **setiap nilai dropout**, masing-masing dengan 5-fold K-Fold (total training jauh lebih
  lama — pastikan GPU aktif).
- Nama file model menyertakan nilai dropout, misal:
  `aspect_model_dropout_0_2_fold_1.keras`, `sentiment_model_<aspek>_dropout_0_5_fold_3.keras`.
- Cell evaluasi menampilkan **perbandingan akurasi per nilai dropout** (per fold, rata-rata
  K-Fold, dan ensemble soft-voting).
- Model terbaik yang disimpan ke `best_models/` dan `best_sentiment_models/` dipilih dari
  akurasi tertinggi **lintas semua nilai dropout & fold**.


#SAVE DATA

In [1]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"

In [2]:
import os
from IPython.display import display  # sudah otomatis tersedia di notebook Kaggle, import ini jaga-jaga saja

# ==========================================================
# KONFIGURASI PATH UNTUK KAGGLE
# ==========================================================
# Dataset 'new-dataset' berisi SEMUA file (xlsx & pkl) LANGSUNG di root-nya,
# TIDAK ADA subfolder apapun (tidak ada 'preprocessed_data', tidak ada
# 'processed-data', tidak ada 'experiment_states', dst). Semua file dibaca
# langsung dari INPUT_DIR.

_KAGGLE_INPUT_ROOT = '/kaggle/input'
_PREFERRED_SLUG = 'new-dataset'

if os.path.isdir(_KAGGLE_INPUT_ROOT):
    _available_datasets = sorted(os.listdir(_KAGGLE_INPUT_ROOT))
else:
    _available_datasets = []

if _PREFERRED_SLUG in _available_datasets:
    INPUT_DIR = os.path.join(_KAGGLE_INPUT_ROOT, _PREFERRED_SLUG)
elif _available_datasets:
    # Fallback: kalau slug persis 'new-dataset' tidak ketemu, otomatis pakai
    # dataset pertama yang tersedia supaya notebook tetap bisa jalan.
    INPUT_DIR = os.path.join(_KAGGLE_INPUT_ROOT, _available_datasets[0])
    print(f"Peringatan: slug '{_PREFERRED_SLUG}' tidak ditemukan di /kaggle/input, "
          f"otomatis pakai dataset pertama yang tersedia: '{_available_datasets[0]}'")
else:
    INPUT_DIR = os.path.join(_KAGGLE_INPUT_ROOT, _PREFERRED_SLUG)
    print("Peringatan: /kaggle/input kosong atau tidak ada. Pastikan dataset sudah "
          "ditambahkan lewat '+ Add Input' lalu restart session.")

# WORK_DIR -> folder kerja Kaggle, tempat semua output (model, pickle,
# dataset hasil olahan) disimpan. Isinya otomatis ikut tersimpan saat
# kamu klik "Save Version".
WORK_DIR = '/kaggle/working'
os.makedirs(WORK_DIR, exist_ok=True)

# DROPOUT_VALUES -> daftar nilai dropout yang ingin diuji coba pada
# training LSTM (klasifikasi aspek maupun sentimen per aspek).
DROPOUT_VALUES = [0.2, 0.5, 0.8]

# ==========================================================
# HELPER: cari file di dalam dataset
# ==========================================================
def find_input_file(filename, base_dir=None, required=True):
    """Cari `filename` di dalam base_dir (default INPUT_DIR).
    Dataset SEHARUSNYA flat (file langsung ada di root), tapi helper ini
    tetap mencari satu tingkat ke bawah untuk jaga-jaga. Kalau tidak
    ketemu, isi folder dataset ditampilkan supaya gampang di-debug."""
    base_dir = base_dir or INPUT_DIR

    direct_path = os.path.join(base_dir, filename)
    if os.path.isfile(direct_path):
        return direct_path

    if os.path.isdir(base_dir):
        for root, _dirs, files in os.walk(base_dir):
            if filename in files:
                return os.path.join(root, filename)

    if required:
        print(f"File '{filename}' tidak ditemukan di dalam '{base_dir}'.")
        if os.path.isdir(base_dir):
            print("Isi folder dataset saat ini:")
            found_any = False
            for root, _dirs, files in os.walk(base_dir):
                for f in files:
                    found_any = True
                    print(" -", os.path.relpath(os.path.join(root, f), base_dir))
            if not found_any:
                print(" (folder kosong)")
        else:
            print(f"Folder '{base_dir}' tidak ada sama sekali.")
        raise FileNotFoundError(
            f"'{filename}' tidak ditemukan di '{base_dir}'. Pastikan dataset "
            f"'{_PREFERRED_SLUG}' sudah ditambahkan via '+ Add Input', berisi file ini "
            f"langsung di root, dan session sudah di-restart setelah menambahkannya."
        )
    return None

print(f"Dataset yang terdeteksi di /kaggle/input : {_available_datasets}")
print(f"INPUT_DIR (dataset flat, semua file di root) : {INPUT_DIR}")
print(f"WORK_DIR  (folder kerja / output)             : {WORK_DIR}")
print(f"DROPOUT_VALUES (uji coba dropout)             : {DROPOUT_VALUES}")


Peringatan: slug 'new-dataset' tidak ditemukan di /kaggle/input, otomatis pakai dataset pertama yang tersedia: 'datasets'
Dataset yang terdeteksi di /kaggle/input : ['datasets']
INPUT_DIR (dataset flat, semua file di root) : /kaggle/input/datasets
WORK_DIR  (folder kerja / output)             : /kaggle/working
DROPOUT_VALUES (uji coba dropout)             : [0.2, 0.5, 0.8]


In [3]:
import os

# Sebelumnya ada DUA folder root berbeda (BASE_DRIVE_PATH = 'epoch_lima' dan
# SAVE_ROOT_PATH = 'EPOCH 05'), sehingga model, tokenizer, dan dataset olahan
# tersebar di dua tempat berbeda. Sekarang disatukan jadi SATU folder saja:
# WORK_DIR/EPOCH_15 -- dipakai bersama oleh BASE_DRIVE_PATH & SAVE_ROOT_PATH.
BASE_DRIVE_PATH = os.path.join(WORK_DIR, 'EPOCH_20')
SAVE_ROOT_PATH = BASE_DRIVE_PATH  # alias, disatukan supaya tidak ada folder duplikat

os.makedirs(BASE_DRIVE_PATH, exist_ok=True)
os.makedirs(os.path.join(BASE_DRIVE_PATH, 'experiment_states'), exist_ok=True)

print(f"Base path (disatukan) untuk semua output disetel ke: {BASE_DRIVE_PATH}")


Base path (disatukan) untuk semua output disetel ke: /kaggle/working/EPOCH_20


In [4]:
import pandas as pd
import os

# Dataset 'new-dataset' FLAT -> file langsung ada di root INPUT_DIR (tidak ada
# subfolder 'preprocessed_data' / 'processed-data' sama sekali).
# PREPROCESSED_DATA_DIR dipertahankan sebagai alias supaya cell-cell
# berikutnya tidak perlu diubah strukturnya.
PREPROCESSED_DATA_DIR = INPUT_DIR
# CATATAN: /kaggle/input bersifat READ-ONLY -> tidak perlu (dan tidak bisa) os.makedirs di sini.

try:
    file_path_to_load = find_input_file("dataset_hasil_labelling_aspek.xlsx", base_dir=PREPROCESSED_DATA_DIR)
    print(f"Mencoba memuat file dari: {file_path_to_load}")

    df_loaded = pd.read_excel(file_path_to_load)

    print("File berhasil dimuat! Berikut adalah 5 baris pertama:")
    display(df_loaded.head())

except FileNotFoundError as e:
    print(f"Error: {e}")
except Exception as e:
    print(f"Terjadi kesalahan saat memuat file: {e}")


Mencoba memuat file dari: /kaggle/input/datasets/mellychanwardani/new-dataset/dataset_hasil_labelling_aspek.xlsx
File berhasil dimuat! Berikut adalah 5 baris pertama:


,Original_Review_ID,Review,Extracted_Opinion,Translated_Opinion,Cleaned_Opinion,Casefolded_Opinion,Tokenized_Opinion,Vader_Sentiment_Category,Sentiment_encoded,Aspek,Aspek_encoded
0,0,Ruang flamboyan rumah sakit mitra keluarga dar...,kamar penuh,full room,kamar penuh,kamar penuh,"['kamar', 'penuh']",Neutral,1,fasilitas dan infrastruktur,1
1,1,Ruang flamboyan rumah sakit mitra keluarga dar...,perawat ramah,friendly nurse,perawat ramah,perawat ramah,"['perawat', 'ramah']",Positive,2,kualitas pelayanan medis dan staf,2
2,2,nilai plusnya -dekat dari rumah -obat yang dib...,kebetulan cocok,coincidentally matched,kebetulan cocok,kebetulan cocok,"['kebetulan', 'cocok']",Neutral,1,lainnya,3
3,3,Pelayanan baik memuaskan,Pelayanan baik,Good service,Pelayanan baik,pelayanan baik,"['pelayanan', 'baik']",Positive,2,kualitas pelayanan medis dan staf,2
4,4,"Dirawat inap di kamar Flamboyan, setelah opera...",tenaga medis,medical personnel,tenaga medis,tenaga medis,"['tenaga', 'medis']",Neutral,1,kualitas pelayanan medis dan staf,2


In [5]:
import os
import pandas as pd

# Ensure SAVE_ROOT_PATH is defined (it should be from previous cells)
if 'SAVE_ROOT_PATH' not in globals():
    SAVE_ROOT_PATH = os.path.join(WORK_DIR, 'EPOCH_20')  # Fallback value if not defined (disatukan dengan BASE_DRIVE_PATH)
    print("Warning: SAVE_ROOT_PATH not found, using default fallback path.")

# Define the directory where you want to save the file
save_directory = SAVE_ROOT_PATH

# Ensure the save directory exists
os.makedirs(save_directory, exist_ok=True)

# Define the full path for the output file
output_file_path = os.path.join(save_directory, 'dataset_hasil_labelling_aspek.xlsx')

# Check if df_loaded exists and is a DataFrame before saving
if 'df_loaded' in globals() and isinstance(df_loaded, pd.DataFrame):
    try:
        # Save the DataFrame to an Excel file
        df_loaded.to_excel(output_file_path, index=False)
        print(f"✅ Dataset 'dataset_hasil_labelling_aspek' berhasil disimpan ke: {output_file_path}")
    except Exception as e:
        print(f"❌ Terjadi kesalahan saat menyimpan file: {e}")
else:
    print("❌ DataFrame 'df_loaded' tidak ditemukan atau bukan DataFrame. Pastikan telah dimuat sebelumnya.")


✅ Dataset 'dataset_hasil_labelling_aspek' berhasil disimpan ke: /kaggle/working/EPOCH_20/dataset_hasil_labelling_aspek.xlsx


In [6]:
!pip install -q scikit-learn==1.8.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 22.7 MB/s eta 0:00:0000:0100:01


In [7]:
import os
import pickle
from sklearn.preprocessing import LabelEncoder  # Import LabelEncoder for potential fallback initialization

# aspect_le.pkl & sentiment_le.pkl diharapkan ada LANGSUNG di root dataset
# 'new-dataset' (flat, tanpa subfolder 'experiment_states').
EXPERIMENT_STATES_DIR = INPUT_DIR  # alias, dipertahankan untuk kompatibilitas
# CATATAN: /kaggle/input bersifat READ-ONLY -> tidak perlu os.makedirs di sini.

# --- Load aspect_le.pkl ---
aspect_le = None  # Initialize aspect_le
try:
    aspect_le_path = find_input_file('aspect_le.pkl', base_dir=EXPERIMENT_STATES_DIR)
    with open(aspect_le_path, 'rb') as f:
        aspect_le = pickle.load(f)
    print(f"'aspect_le.pkl' berhasil dimuat dari: {aspect_le_path}")
    print(f"Kelas aspek yang dimuat: {aspect_le.classes_}")
except FileNotFoundError as e:
    print(f"Error: {e}")
    # Optionally, initialize with a dummy LabelEncoder if file not found
    aspect_le = LabelEncoder()
except Exception as e:
    print(f"Terjadi kesalahan saat memuat 'aspect_le.pkl': {e}")
    aspect_le = LabelEncoder()

# --- Load sentiment_le.pkl ---
sentiment_le = None  # Initialize sentiment_le
try:
    sentiment_le_path = find_input_file('sentiment_le.pkl', base_dir=EXPERIMENT_STATES_DIR)
    with open(sentiment_le_path, 'rb') as f:
        sentiment_le = pickle.load(f)
    print(f"'sentiment_le.pkl' berhasil dimuat dari: {sentiment_le_path}")
    print(f"Kelas sentimen yang dimuat: {sentiment_le.classes_}")
except FileNotFoundError as e:
    print(f"Error: {e}")
    # Optionally, initialize with a dummy LabelEncoder if file not found
    sentiment_le = LabelEncoder()
except Exception as e:
    print(f"Terjadi kesalahan saat memuat 'sentiment_le.pkl': {e}")
    sentiment_le = LabelEncoder()


'aspect_le.pkl' berhasil dimuat dari: /kaggle/input/datasets/mellychanwardani/new-dataset/aspect_le.pkl
Kelas aspek yang dimuat: ['biaya layanan' 'fasilitas dan infrastruktur'
 'kualitas pelayanan medis dan staf' 'lainnya' 'waktu tunggu']
'sentiment_le.pkl' berhasil dimuat dari: /kaggle/input/datasets/mellychanwardani/new-dataset/sentiment_le.pkl
Kelas sentimen yang dimuat: ['Negative' 'Neutral' 'Positive']


In [8]:
import pickle
import os

# Ensure EXPERIMENT_DIR_FULL_PATH is defined and correct
if 'EXPERIMENT_DIR_FULL_PATH' not in globals():
    # Fallback if EXPERIMENT_DIR_FULL_PATH somehow wasn't defined
    SAVE_ROOT_PATH = os.path.join(WORK_DIR, 'EPOCH_20')
    EXPERIMENT_DIR_FULL_PATH = os.path.join(SAVE_ROOT_PATH, 'experiment_states')
    os.makedirs(EXPERIMENT_DIR_FULL_PATH, exist_ok=True)
    print("Warning: EXPERIMENT_DIR_FULL_PATH not defined, using default fallback path.")

# Save aspect_le.pkl
if 'aspect_le' in globals():
    aspect_le_save_path = os.path.join(EXPERIMENT_DIR_FULL_PATH, 'aspect_le.pkl')
    with open(aspect_le_save_path, "wb") as f:
        pickle.dump(aspect_le, f)
    print(f"'aspect_le.pkl' saved successfully to: '{aspect_le_save_path}'")
else:
    print("Error: 'aspect_le' object not found in current session. Not saving aspect_le.pkl.")

# Save sentiment_le.pkl
if 'sentiment_le' in globals():
    sentiment_le_save_path = os.path.join(EXPERIMENT_DIR_FULL_PATH, 'sentiment_le.pkl')
    with open(sentiment_le_save_path, "wb") as f:
        pickle.dump(sentiment_le, f)
    print(f"'sentiment_le.pkl' saved successfully to: '{sentiment_le_save_path}'")
else:
    print("Error: 'sentiment_le' object not found in current session. Not saving sentiment_le.pkl.")


'aspect_le.pkl' saved successfully to: '/kaggle/working/EPOCH_20/experiment_states/aspect_le.pkl'
'sentiment_le.pkl' saved successfully to: '/kaggle/working/EPOCH_20/experiment_states/sentiment_le.pkl'


#KLASIFIKASI ASPEK

In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split
import os

# Ensure PREPROCESSED_DATA_DIR is defined
# (It should be defined in previous cells, but we ensure its availability)
if 'PREPROCESSED_DATA_DIR' not in globals():
    # Dataset Kaggle diupload FLAT (tanpa subfolder) -> langsung pakai INPUT_DIR
    print("Warning: PREPROCESSED_DATA_DIR not found, falling back to INPUT_DIR.")
    PREPROCESSED_DATA_DIR = INPUT_DIR

df_data = pd.read_excel(find_input_file('dataset_hasil_labelling_aspek.xlsx', base_dir=PREPROCESSED_DATA_DIR))
X_data = df_data[['Tokenized_Opinion']]
Y_aspect_data = df_data['Aspek_encoded']
Y_sentiment_data = df_data['Sentiment_encoded']

X_train_full, X_test_full, Y_train_aspect, Y_test_aspect = train_test_split(
    X_data,
    Y_aspect_data,
    test_size=0.2,
    random_state=42,
    stratify=Y_aspect_data
)

Y_train_sent = Y_sentiment_data.loc[X_train_full.index]
Y_test_sent = Y_sentiment_data.loc[X_test_full.index]

print("--- Hasil Split ---")
print("Data Training:", X_train_full.shape[0], "baris")
print("Data Testing:", X_test_full.shape[0], "baris")

--- Hasil Split ---
Data Training: 4204 baris
Data Testing: 1051 baris


In [10]:
import numpy as np
import pandas as pd
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import ast
import os
import pickle

def ensure_list(text):
    if isinstance(text, str) and text.startswith('['):
        try:
            return ast.literal_eval(text)
        except (ValueError, SyntaxError):
            return []
    elif isinstance(text, list):
        return text
    return []

X_train_texts = X_train_full['Tokenized_Opinion'].apply(ensure_list).apply(lambda x: ' '.join(x) if x else '')
X_test_texts = X_test_full['Tokenized_Opinion'].apply(ensure_list).apply(lambda x: ' '.join(x) if x else '')

print(f"  Training samples (X_train_texts): {len(X_train_texts)}")
print(f"  Testing samples (X_test_texts): {len(X_test_texts)}")
print(f"  Example X_train_texts (first row): {X_train_texts.head(1).iloc[0]}")

# Tokenizer
tokenizer = Tokenizer(num_words=5000, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train_texts)  # fit only on training data texts

# Convert texts to sequences of indices
X_train_seq = tokenizer.texts_to_sequences(X_train_texts)
X_test_seq = tokenizer.texts_to_sequences(X_test_texts)

# Padding
maxlen = 20  # Max length for sequences
X_train_pad = pad_sequences(X_train_seq, maxlen=maxlen, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=maxlen, padding='post', truncating='post')

# Info results
print("Tokenization & Padding complete.")
print(f"  Vocabulary size   : {len(tokenizer.word_index)} unique words")
print(f"  Train data shape  : {X_train_pad.shape}")
print(f"  Test data shape   : {X_test_pad.shape}")

if X_train_seq:
    print(f"  Example first train sequence (first 20 tokens):\n{X_train_seq[0][:20]}")
else:
    print("  No training sequences generated.")

# Save tokenizer ke WORK_DIR (Kaggle)
# SAVE_ROOT_PATH sudah disatukan dengan BASE_DRIVE_PATH di cell konfigurasi
# (WORK_DIR/EPOCH_05), jadi tidak perlu didefinisikan ulang ke folder lain.
if 'SAVE_ROOT_PATH' not in globals():
    SAVE_ROOT_PATH = os.path.join(WORK_DIR, 'EPOCH_05')
EXPERIMENT_DIR_FULL_PATH = os.path.join(SAVE_ROOT_PATH, 'experiment_states')

os.makedirs(EXPERIMENT_DIR_FULL_PATH, exist_ok=True)
tokenizer_save_path = os.path.join(EXPERIMENT_DIR_FULL_PATH, 'tokenizer_lstm.pkl')
with open(tokenizer_save_path, "wb") as f:
    pickle.dump(tokenizer, f)
print(f"Tokenizer saved to '{tokenizer_save_path}'")

  Training samples (X_train_texts): 4204
  Testing samples (X_test_texts): 1051
  Example X_train_texts (first row): sakit sebelah perlu rujukan
Tokenization & Padding complete.
  Vocabulary size   : 1503 unique words
  Train data shape  : (4204, 20)
  Test data shape   : (1051, 20)
  Example first train sequence (first 20 tokens):
[3, 380, 241, 323]
Tokenizer saved to '/kaggle/working/EPOCH_20/experiment_states/tokenizer_lstm.pkl'


In [11]:
import pickle
from sklearn.preprocessing import LabelEncoder

# Load the aspect label encoder from the drive
aspect_le_path = os.path.join(EXPERIMENT_DIR_FULL_PATH, 'aspect_le.pkl') # Corrected: consistency with 'sentiment_le.pkl' to match global testing

try:
    with open(aspect_le_path, 'rb') as f:
        aspect_le = pickle.load(f)
    print(f"Aspect LabelEncoder loaded from '{aspect_le_path}'")
    print(f"Number of aspect classes: {len(aspect_le.classes_)}")
except FileNotFoundError:
    print(f"Error: '{aspect_le_path}' not found. Please ensure the aspect label encoder is saved at this location.")
    aspect_le = LabelEncoder()
    if 'df_data' in globals():
        aspect_le.fit(df_data['Aspek'])
        print("Warning: Aspect LabelEncoder not found, a new one was created from df_data['Aspek']. Please ensure this is the intended behavior.")
        print(f"Number of aspect classes (newly created): {len(aspect_le.classes_)}")
    else:
        print("Error: df_data not found to create aspect_le. Please check previous cells.")

# Load or create the sentiment label encoder
sentiment_le_path = os.path.join(EXPERIMENT_DIR_FULL_PATH, 'sentiment_le.pkl') # Corrected: consistent naming with saving in Lx5RY78R5SW4 and loading in f11050e1

try:
    with open(sentiment_le_path, 'rb') as f:
        sentiment_le = pickle.load(f)
    print(f"Sentiment LabelEncoder loaded from '{sentiment_le_path}'")
    print(f"Sentiment classes: {sentiment_le.classes_}")
except FileNotFoundError:
    print(f"Error: '{sentiment_le_path}' not found. Please ensure the sentiment label encoder is saved at this location.")
    sentiment_le = LabelEncoder()
    if 'df_data' in globals():
        sentiment_le.fit(df_data['Vader_Sentiment_Category'])
        print("Warning: Sentiment LabelEncoder not found, a new one was created from df_data['Vader_Sentiment_Category']. Please ensure this is the intended behavior.")
        print(f"Sentiment classes (newly created): {sentiment_le.classes_}")
        # Optionally save it for future runs
        try:
            with open(sentiment_le_path, "wb") as f:
                pickle.dump(sentiment_le, f)
            print(f"Newly created Sentiment LabelEncoder saved to '{sentiment_le_path}'")
        except Exception as e:
            print(f"Could not save sentiment_le to drive: {e}")
    else:
        print("Error: df_data not found to create sentiment_le. Please check previous cells.")

Aspect LabelEncoder loaded from '/kaggle/working/EPOCH_20/experiment_states/aspect_le.pkl'
Number of aspect classes: 5
Sentiment LabelEncoder loaded from '/kaggle/working/EPOCH_20/experiment_states/sentiment_le.pkl'
Sentiment classes: ['Negative' 'Neutral' 'Positive']


In [12]:
from sklearn.model_selection import KFold
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
# from tensorflow.keras.callbacks import EarlyStopping # Removed EarlyStopping

import numpy as np
import tensorflow as tf
import random
import os # Added os import for saving models correctly

# =========================
# RANDOM SEED
# =========================
SEED = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)

# =========================
# PARAMETER
# =========================
vocab_size = 5000
max_len = 20
num_classes = len(aspect_le.classes_)

EPOCHS = 20

# =========================
# PARAMETER UJI COBA DROPOUT
# =========================
# DROPOUT_VALUES didefinisikan di cell konfigurasi paling atas.
# Kalau belum ada (misal cell konfigurasi belum dijalankan), fallback ke default berikut.
if 'DROPOUT_VALUES' not in globals():
    DROPOUT_VALUES = [0.2, 0.5, 0.8]

print(f"Dropout yang akan diuji untuk klasifikasi aspek: {DROPOUT_VALUES}")

# =========================
# KFOLD
# =========================
skf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

# Define directory for saving models
model_save_dir = os.path.join(BASE_DRIVE_PATH, 'aspect_classification_models')
os.makedirs(model_save_dir, exist_ok=True)

# Menyimpan ringkasan rata-rata akurasi per nilai dropout
dropout_summary = []

# =========================
# LOOP TIAP NILAI DROPOUT
# =========================
for dropout_rate in DROPOUT_VALUES:

    dropout_tag = str(dropout_rate).replace('.', '_')

    print("\n" + "#"*60)
    print(f"# TRAINING KLASIFIKASI ASPEK DENGAN DROPOUT = {dropout_rate}")
    print("#"*60)

    print("="*60)
    print(f"TRAINING DENGAN MAX EPOCH = {EPOCHS}")
    print("="*60)

    # best_val_loss = float('inf') # Removed as EarlyStopping is removed
    fold_no = 1

    fold_accuracies = []

    # =========================
    # KFOLD LOOP
    # =========================
    for train_index, val_index in skf.split(X_train_pad, Y_train_aspect):

        print(f"\n====== Dropout {dropout_rate} | Fold {fold_no} ======")

        # =========================
        # SPLIT DATA
        # =========================
        X_tr = X_train_pad[train_index]
        X_val = X_train_pad[val_index]

        y_tr = Y_train_aspect.iloc[train_index]
        y_val = Y_train_aspect.iloc[val_index]

        # =========================
        # MODEL
        # =========================
        model = Sequential([

            Embedding(
                input_dim=vocab_size,
                output_dim=128,
                mask_zero=True
            ),

            LSTM(
                128,
                return_sequences=True,
                dropout=dropout_rate,
                recurrent_dropout=dropout_rate
            ),

            LSTM(
                64,
                dropout=dropout_rate,
                recurrent_dropout=dropout_rate
            ),

            Dense(128, activation='relu'),

            Dropout(dropout_rate),

            Dense(num_classes, activation='softmax')
        ])

        # =========================
        # COMPILE
        # =========================
        model.compile(
            optimizer=Adam(learning_rate=0.0005),
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy']
        )

        # =========================
        # EARLY STOPPING - REMOVED
        # =========================
        # early_stopping = EarlyStopping(
        #     monitor='val_loss',
        #     patience=2,
        #     restore_best_weights=True,
        #     verbose=1
        # )

        # =========================
        # TRAINING
        # =========================
        history = model.fit(
            X_tr,
            y_tr,
            epochs=EPOCHS,
            batch_size=32,
            validation_data=(X_val, y_val),
            # callbacks=[early_stopping], # Removed early_stopping from callbacks
            verbose=1
        )

        # =========================
        # EVALUASI
        # =========================
        scores = model.evaluate(
            X_val,
            y_val,
            verbose=0
        )

        print(f"\nDropout {dropout_rate} | Fold {fold_no}")
        print(f"Loss     : {scores[0]:.4f}")
        print(f"Accuracy : {scores[1]:.4f}")

        fold_accuracies.append(scores[1])

        # =========================
        # SAVE MODEL FOR EACH FOLD
        # =========================
        # Nama file sekarang menyertakan nilai dropout supaya tiap konfigurasi
        # bisa dibandingkan pada cell evaluasi berikutnya.
        model_filename = os.path.join(
            model_save_dir,
            f"aspect_model_dropout_{dropout_tag}_fold_{fold_no}.keras"
        )
        model.save(model_filename)
        print(f"Model for Dropout {dropout_rate} Fold {fold_no} saved to: {model_filename}")

        fold_no += 1

    # =========================
    # RATA-RATA AKURASI UNTUK DROPOUT INI
    # =========================
    avg_acc = np.mean(fold_accuracies)
    dropout_summary.append({'dropout': dropout_rate, 'avg_accuracy': avg_acc})

    print("\n" + "="*60)
    print(f"HASIL AKHIR DROPOUT = {dropout_rate}")
    print("="*60)

    print(f"Average Accuracy : {avg_acc:.4f}")

# =========================
# RINGKASAN PERBANDINGAN SEMUA DROPOUT
# =========================
print("\n" + "#"*60)
print("# RINGKASAN PERBANDINGAN DROPOUT (Klasifikasi Aspek)")
print("#"*60)
for row in dropout_summary:
    print(f"Dropout {row['dropout']}: rata-rata accuracy (K-Fold) = {row['avg_accuracy']:.4f}")


Dropout yang akan diuji untuk klasifikasi aspek: [0.2, 0.5, 0.8]

############################################################
# TRAINING KLASIFIKASI ASPEK DENGAN DROPOUT = 0.2
############################################################
TRAINING DENGAN MAX EPOCH = 20

====== Dropout 0.2 | Fold 1 ======
Epoch 1/20


I0000 00:00:1782917319.881454      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1782917319.884431      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


106/106 ━━━━━━━━━━━━━━━━━━━━ 27s 139ms/step - accuracy: 0.6408 - loss: 1.1681 - val_accuracy: 0.8098 - val_loss: 0.6995
Epoch 2/20
106/106 ━━━━━━━━━━━━━━━━━━━━ 14s 130ms/step - accuracy: 0.8891 - loss: 0.4208 - val_accuracy: 0.9144 - val_loss: 0.3001
Epoch 3/20
106/106 ━━━━━━━━━━━━━━━━━━━━ 13s 126ms/step - accuracy: 0.9575 - loss: 0.1535 - val_accuracy: 0.9489 - val_loss: 0.1861
Epoch 4/20
106/106 ━━━━━━━━━━━━━━━━━━━━ 14s 128ms/step - accuracy: 0.9712 - loss: 0.0843 - val_accuracy: 0.9560 - val_loss: 0.1463
Epoch 5/20
106/106 ━━━━━━━━━━━━━━━━━━━━ 14s 129ms/step - accuracy: 0.9786 - loss: 0.0515 - val_accuracy: 0.9631 - val_loss: 0.1198
Epoch 6/20
106/106 ━━━━━━━━━━━━━━━━━━━━ 14s 129ms/step - accuracy: 0.9875 - loss: 0.0361 - val_accuracy: 0.9727 - val_loss: 0.1003
Epoch 7/20
106/106 ━━━━━━━━━━━━━━━━━━━━ 14s 128ms/step - accuracy: 0.9932 - loss: 0.0254 - val_accuracy: 0.9774 - val_loss: 0.0786
Epoch 8/20
106/106 ━━━━━━━━━━━━━━━━━━━━ 14s 129ms/step - accuracy: 0.9955 - loss: 0.0168 - val

In [13]:
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

import numpy as np
import pandas as pd
import os
import re
import ast

# =========================
# FUNCTION
# =========================

def ensure_list(x):

    if isinstance(x, list):
        return x

    if isinstance(x, str):

        try:
            val = ast.literal_eval(x)

            if isinstance(val, list):
                return val

        except:
            return x.split()

    return []

# =========================
# PREPARE TEST DATA
# =========================

tokens = X_test_full['Tokenized_Opinion'].apply(ensure_list)

X_test_texts = tokens.apply(lambda x: ' '.join(x)).fillna("")

print("Contoh teks:")
print(X_test_texts.iloc[0])

X_test_seq = tokenizer.texts_to_sequences(X_test_texts)

X_test_pad_final = pad_sequences(
    X_test_seq,
    maxlen=max_len,
    padding='post',
    truncating='post'
)

y_true = np.array(Y_test_aspect)

print("\nShape X_test:", X_test_pad_final.shape)
print("Shape y_test:", y_true.shape)

# =========================
# LOAD MODEL FILES
# =========================

model_save_dir = os.path.join(
    BASE_DRIVE_PATH,
    'aspect_classification_models'
)

model_files = [
    f for f in os.listdir(model_save_dir)
    if f.endswith('.keras')
    and f.startswith('aspect_model_dropout_')
]

def parse_dropout_fold(filename):
    """Ambil nilai dropout & nomor fold dari nama file aspect_model_dropout_<d>_fold_<n>.keras"""
    match = re.search(r'dropout_([0-9_]+)_fold_(\d+)\.keras', filename)
    if match:
        dropout_value = float(match.group(1).replace('_', '.'))
        fold_number = int(match.group(2))
        return dropout_value, fold_number
    return None, -1

model_files = sorted(
    model_files,
    key=lambda f: parse_dropout_fold(f)
)

results = []
# Prediksi dikelompokkan per nilai dropout supaya bisa dibuat ensemble per dropout
preds_by_dropout = {}

if not model_files:

    print(
        f"\nTidak ada model ditemukan di {model_save_dir}"
    )

else:

    print("\nModel ditemukan:")
    for file in model_files:
        print("-", file)

    for file_name in model_files:

        full_file_path = os.path.join(
            model_save_dir,
            file_name
        )

        dropout_value, fold_number = parse_dropout_fold(file_name)

        print("\n" + "="*60)
        print(
            f"EVALUASI MODEL | Dropout {dropout_value} | Fold {fold_number}"
        )
        print("="*60)

        model = load_model(full_file_path)

        y_pred_prob = model.predict(
            X_test_pad_final,
            verbose=0
        )

        preds_by_dropout.setdefault(dropout_value, []).append(y_pred_prob)

        y_pred = np.argmax(
            y_pred_prob,
            axis=1
        )

        print("\nDistribusi Prediksi:")
        print(np.bincount(y_pred))

        # =========================
        # METRICS
        # =========================

        acc = accuracy_score(
            y_true,
            y_pred
        )

        precision = precision_score(
            y_true,
            y_pred,
            average='weighted',
            zero_division=0
        )

        recall = recall_score(
            y_true,
            y_pred,
            average='weighted',
            zero_division=0
        )

        f1 = f1_score(
            y_true,
            y_pred,
            average='weighted',
            zero_division=0
        )

        print(f"\nAccuracy      : {acc:.4f}")
        print(f"Precision     : {precision:.4f}")
        print(f"Recall        : {recall:.4f}")
        print(f"F1-Score      : {f1:.4f}")

        results.append({
            "model_name": file_name,
            "dropout": dropout_value,
            "fold": fold_number,
            "accuracy": acc,
            "precision": precision,
            "recall": recall,
            "f1_score": f1
        })

# =========================
# HASIL SEMUA MODEL (per fold, semua dropout)
# =========================

df_results = pd.DataFrame(results)

if not df_results.empty:

    df_results = df_results.sort_values(
        by='accuracy',
        ascending=False
    )

    print("\n" + "="*60)
    print("PERBANDINGAN HASIL MODEL (semua dropout & fold)")
    print("="*60)

    display(df_results)

    # =========================
    # RINGKASAN RATA-RATA PER NILAI DROPOUT
    # =========================
    print("\n" + "="*60)
    print("RATA-RATA AKURASI PER NILAI DROPOUT")
    print("="*60)

    df_dropout_summary = (
        df_results
        .groupby('dropout')[['accuracy', 'precision', 'recall', 'f1_score']]
        .mean()
        .sort_values(by='accuracy', ascending=False)
    )
    display(df_dropout_summary)

# =========================
# ENSEMBLE PER NILAI DROPOUT (soft voting antar fold, per dropout)
# =========================

ensemble_summary = []

for dropout_value, prob_list in preds_by_dropout.items():

    avg_y_pred_prob = np.mean(prob_list, axis=0)
    y_pred_ensemble = np.argmax(avg_y_pred_prob, axis=1)

    ensemble_accuracy = accuracy_score(y_true, y_pred_ensemble)
    ensemble_precision = precision_score(y_true, y_pred_ensemble, average='weighted', zero_division=0)
    ensemble_recall = recall_score(y_true, y_pred_ensemble, average='weighted', zero_division=0)
    ensemble_f1 = f1_score(y_true, y_pred_ensemble, average='weighted', zero_division=0)

    ensemble_summary.append({
        'dropout': dropout_value,
        'ensemble_accuracy': ensemble_accuracy,
        'ensemble_precision': ensemble_precision,
        'ensemble_recall': ensemble_recall,
        'ensemble_f1': ensemble_f1
    })

    print("\n" + "="*60)
    print(f"EVALUASI ENSEMBLE (soft voting 5-fold) | Dropout = {dropout_value}")
    print("="*60)

    print(f"Accuracy      : {ensemble_accuracy:.4f}")
    print(f"Precision     : {ensemble_precision:.4f}")
    print(f"Recall        : {ensemble_recall:.4f}")
    print(f"F1-Score      : {ensemble_f1:.4f}")

    print("\nClassification Report:")
    if 'aspect_le' in globals():
        target_names_str = [str(cls) for cls in aspect_le.classes_]
        print(classification_report(y_true, y_pred_ensemble, target_names=target_names_str, zero_division=0))
    else:
        print(classification_report(y_true, y_pred_ensemble, zero_division=0))

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred_ensemble))

if ensemble_summary:
    print("\n" + "#"*60)
    print("# RINGKASAN ENSEMBLE PER DROPOUT (Klasifikasi Aspek)")
    print("#"*60)
    df_ensemble_summary = pd.DataFrame(ensemble_summary).sort_values(by='ensemble_accuracy', ascending=False)
    display(df_ensemble_summary)


Contoh teks:
instruksi jelas

Shape X_test: (1051, 20)
Shape y_test: (1051,)

Model ditemukan:
- aspect_model_dropout_0_2_fold_1.keras
- aspect_model_dropout_0_2_fold_2.keras
- aspect_model_dropout_0_2_fold_3.keras
- aspect_model_dropout_0_2_fold_4.keras
- aspect_model_dropout_0_2_fold_5.keras
- aspect_model_dropout_0_5_fold_1.keras
- aspect_model_dropout_0_5_fold_2.keras
- aspect_model_dropout_0_5_fold_3.keras
- aspect_model_dropout_0_5_fold_4.keras
- aspect_model_dropout_0_5_fold_5.keras
- aspect_model_dropout_0_8_fold_1.keras
- aspect_model_dropout_0_8_fold_2.keras
- aspect_model_dropout_0_8_fold_3.keras
- aspect_model_dropout_0_8_fold_4.keras
- aspect_model_dropout_0_8_fold_5.keras

EVALUASI MODEL | Dropout 0.2 | Fold 1

Distribusi Prediksi:
[ 16 181 377 424  53]

Accuracy      : 0.9914
Precision     : 0.9915
Recall        : 0.9914
F1-Score      : 0.9914

EVALUASI MODEL | Dropout 0.2 | Fold 2

Distribusi Prediksi:
[ 15 180 386 416  54]

Accuracy      : 0.9838
Precision     : 0.9841

,model_name,dropout,fold,accuracy,precision,recall,f1_score
0,aspect_model_dropout_0_2_fold_1.keras,0.2,1,0.991437,0.991455,0.991437,0.991427
7,aspect_model_dropout_0_5_fold_3.keras,0.5,3,0.989534,0.989557,0.989534,0.989524
8,aspect_model_dropout_0_5_fold_4.keras,0.5,4,0.988582,0.988707,0.988582,0.988603
9,aspect_model_dropout_0_5_fold_5.keras,0.5,5,0.987631,0.987608,0.987631,0.987601
5,aspect_model_dropout_0_5_fold_1.keras,0.5,1,0.986679,0.986784,0.986679,0.986688
4,aspect_model_dropout_0_2_fold_5.keras,0.2,5,0.985728,0.985815,0.985728,0.985736
3,aspect_model_dropout_0_2_fold_4.keras,0.2,4,0.985728,0.986041,0.985728,0.985682
6,aspect_model_dropout_0_5_fold_2.keras,0.5,2,0.985728,0.986300,0.985728,0.985838
1,aspect_model_dropout_0_2_fold_2.keras,0.2,2,0.983825,0.984052,0.983825,0.983775
2,aspect_model_dropout_0_2_fold_3.keras,0.2,3,0.982873,0.982906,0.982873,0.982868



RATA-RATA AKURASI PER NILAI DROPOUT


,accuracy,precision,recall,f1_score
dropout,,,,
0.5,0.987631,0.987791,0.987631,0.987650
0.2,0.985918,0.986054,0.985918,0.985898
0.8,0.960038,0.949468,0.960038,0.953919



EVALUASI ENSEMBLE (soft voting 5-fold) | Dropout = 0.2
Accuracy      : 0.9933
Precision     : 0.9934
Recall        : 0.9933
F1-Score      : 0.9933

Classification Report:
                                   precision    recall  f1-score   support

                    biaya layanan       1.00      0.94      0.97        17
      fasilitas dan infrastruktur       0.99      0.99      0.99       181
kualitas pelayanan medis dan staf       0.99      0.99      0.99       377
                          lainnya       0.99      0.99      0.99       424
                     waktu tunggu       0.98      1.00      0.99        52

                         accuracy                           0.99      1051
                        macro avg       0.99      0.98      0.99      1051
                     weighted avg       0.99      0.99      0.99      1051


Confusion Matrix:
[[ 16   0   0   0   1]
 [  0 180   0   1   0]
 [  0   0 375   2   0]
 [  0   1   2 421   0]
 [  0   0   0   0  52]]

EVALUASI ENSEM

,dropout,ensemble_accuracy,ensemble_precision,ensemble_recall,ensemble_f1
0,0.2,0.993340,0.993358,0.993340,0.993330
1,0.5,0.991437,0.991404,0.991437,0.991416
2,0.8,0.963844,0.953108,0.963844,0.957668


In [14]:
import os
from tensorflow.keras.models import load_model

# Ensure df_results exists and is not empty
if 'df_results' in globals() and not df_results.empty:
    # Get the name of the best model (highest accuracy, across ALL dropout & fold combinations)
    best_row = df_results.iloc[0]
    best_model_name = best_row['model_name']

    # Construct the full path to the best model
    # model_save_dir is already defined in the evaluation cell above
    source_best_model_path = os.path.join(model_save_dir, best_model_name)

    # Define the directory to save the best model
    best_models_dir = os.path.join(BASE_DRIVE_PATH, 'best_models')
    os.makedirs(best_models_dir, exist_ok=True)

    # Define the destination path for the best model with the new requested name
    destination_best_model_path = os.path.join(best_models_dir, 'best_model_aspek.keras')

    try:
        # Load the best model
        best_model = load_model(source_best_model_path)

        # Save the best model to the new location
        best_model.save(destination_best_model_path)

        print(
            f"✅ Best aspect classification model '{best_model_name}' "
            f"(Dropout: {best_row.get('dropout', 'N/A')}, Fold: {best_row.get('fold', 'N/A')}, "
            f"Accuracy: {best_row['accuracy']:.4f}) saved successfully to: {destination_best_model_path}"
        )
    except Exception as e:
        print(f"❌ Error saving the best aspect classification model: {e}")
else:
    print("❌ No model results found to determine the best model. Please ensure the evaluation cell ran successfully.")


✅ Best aspect classification model 'aspect_model_dropout_0_2_fold_1.keras' (Dropout: 0.2, Fold: 1, Accuracy: 0.9914) saved successfully to: /kaggle/working/EPOCH_20/best_models/best_model_aspek.keras


#KLASIFIKASI SENTIMEN PER ASPEK

In [15]:
import pandas as pd
from IPython.display import display

if 'df_aspect_labeling' not in locals():
    # Use the already loaded df_loaded DataFrame
    if 'df_loaded' in globals():
        df_aspect_labeling = df_loaded.copy()
        print("Assigned df_loaded to df_aspect_labeling.")
    else:
        print("Error: df_loaded not found. Please ensure 'dataset_hasil_labelling_aspek.xlsx' was loaded successfully in a previous cell.")
        # Fallback to attempt loading if df_loaded is unexpectedly missing
        try:
            # Assuming PREPROCESSED_DATA_DIR is defined
            if 'PREPROCESSED_DATA_DIR' in globals():
                file_path = find_input_file('dataset_hasil_labelling_aspek.xlsx', base_dir=PREPROCESSED_DATA_DIR)
                df_aspect_labeling = pd.read_excel(file_path)
                print(f"Loaded df_aspect_labeling from '{file_path}' as fallback.")
            else:
                print("Error: PREPROCESSED_DATA_DIR not defined for fallback loading.")
                df_aspect_labeling = pd.DataFrame() # Initialize empty
        except FileNotFoundError:
            print("Error: 'dataset_hasil_labelling_aspek.xlsx' not found. Please ensure previous steps ran successfully.")
            df_aspect_labeling = pd.DataFrame() # Initialize empty

# Membuat dictionary untuk menyimpan DataFrame per aspek
data_per_aspek = {}

# Ensure df_aspect_labeling is not empty before proceeding
if not df_aspect_labeling.empty:
    unique_aspects = df_aspect_labeling['Aspek'].unique()

    print("Memisahkan data berdasarkan aspek...")
    for aspek in unique_aspects:
        data_per_aspek[aspek] = df_aspect_labeling[df_aspect_labeling['Aspek'] == aspek].copy()
        print(f"- Aspek '{aspek}': {len(data_per_aspek[aspek])} baris")

    print("Data berhasil dipisahkan per aspek.")

    # Contoh menampilkan data untuk aspek 'kualitas pelayanan medis dan staf'
    if 'kualitas pelayanan medis dan staf' in data_per_aspek:
        print("\nContoh data untuk aspek 'kualitas pelayanan medis dan staf':")
        display(data_per_aspek['kualitas pelayanan medis dan staf'].head())
    else:
        print("Aspek 'kualitas pelayanan medis dan staf' tidak ditemukan dalam data.")
else:
    print("df_aspect_labeling is empty, skipping aspect separation.")

Assigned df_loaded to df_aspect_labeling.
Memisahkan data berdasarkan aspek...
- Aspek 'fasilitas dan infrastruktur': 903 baris
- Aspek 'kualitas pelayanan medis dan staf': 1886 baris
- Aspek 'lainnya': 2119 baris
- Aspek 'waktu tunggu': 261 baris
- Aspek 'biaya layanan': 86 baris
Data berhasil dipisahkan per aspek.

Contoh data untuk aspek 'kualitas pelayanan medis dan staf':


,Original_Review_ID,Review,Extracted_Opinion,Translated_Opinion,Cleaned_Opinion,Casefolded_Opinion,Tokenized_Opinion,Vader_Sentiment_Category,Sentiment_encoded,Aspek,Aspek_encoded
1,1,Ruang flamboyan rumah sakit mitra keluarga dar...,perawat ramah,friendly nurse,perawat ramah,perawat ramah,"['perawat', 'ramah']",Positive,2,kualitas pelayanan medis dan staf,2
3,3,Pelayanan baik memuaskan,Pelayanan baik,Good service,Pelayanan baik,pelayanan baik,"['pelayanan', 'baik']",Positive,2,kualitas pelayanan medis dan staf,2
4,4,"Dirawat inap di kamar Flamboyan, setelah opera...",tenaga medis,medical personnel,tenaga medis,tenaga medis,"['tenaga', 'medis']",Neutral,1,kualitas pelayanan medis dan staf,2
7,7,"RS yang nyaman, pelayan oke. Suster lantai-lan...",flamboyan baik,flamboyant good,flamboyan baik,flamboyan baik,"['flamboyan', 'baik']",Positive,2,kualitas pelayanan medis dan staf,2
8,8,Pelayanan baik cepat tanggap luar biasa baik. ...,Pelayanan baik,Good service,Pelayanan baik,pelayanan baik,"['pelayanan', 'baik']",Positive,2,kualitas pelayanan medis dan staf,2


In [16]:
import pandas as pd
from sklearn.model_selection import train_test_split
import os

if 'data_per_aspek' not in locals():
    try:
        if 'df_aspect_labeling' not in locals():
            # Use the already loaded df_loaded DataFrame
            if 'df_loaded' in globals():
                df_aspect_labeling = df_loaded.copy()
                print("Assigned df_loaded to df_aspect_labeling.")
            else:
                print("Error: df_loaded not found. Please ensure 'dataset_hasil_labelling_aspek.xlsx' was loaded successfully in a previous cell.")
                # Fallback to attempt loading if df_loaded is unexpectedly missing
                if 'PREPROCESSED_DATA_DIR' in globals():
                    file_path = find_input_file('dataset_hasil_labelling_aspek.xlsx', base_dir=PREPROCESSED_DATA_DIR)
                    df_aspect_labeling = pd.read_excel(file_path)
                    print(f"Loaded df_aspect_labeling from '{file_path}' as fallback.")
                else:
                    print("Error: PREPROCESSED_DATA_DIR not defined for fallback loading.")
                    df_aspect_labeling = pd.DataFrame() # Initialize empty

        if not df_aspect_labeling.empty:
            data_per_aspek = {}
            unique_aspects = df_aspect_labeling['Aspek'].unique()
            for aspek in unique_aspects:
                data_per_aspek[aspek] = df_aspect_labeling[df_aspect_labeling['Aspek'] == aspek].copy()
            print("Re-created data_per_aspek dictionary.")
        else:
            print("df_aspect_labeling is empty, cannot re-create data_per_aspek.")

    except FileNotFoundError:
        print("Error: 'dataset_hasil_labelling_aspek.xlsx' not found.")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

# 1. Initialize an empty dictionary to store split data
split_data_per_aspek = {}

print("Performing train-test split for each aspect...")

# 2. Iterate through each aspect and its corresponding DataFrame
if 'data_per_aspek' in locals() and data_per_aspek:
    for aspek_name, df_aspek in data_per_aspek.items():
        if len(df_aspek) == 0:
            print(f"Skipping empty aspect: {aspek_name}")
            continue

        # 3a. Define features (X) and target (Y)
        X = df_aspek['Tokenized_Opinion'] # Corrected column name
        Y = df_aspek['Sentiment_encoded']

        # Use KFold-like split, no stratification as requested
        # Check if there's enough data for splitting
        if len(df_aspek) > 1:
            X_train_aspect, X_test_aspect, Y_train_sentiment, Y_test_sentiment = train_test_split(
                X,
                Y,
                test_size=0.2,
                random_state=42
            )
            print(f"- Aspect '{aspek_name}': Data split. Train samples: {len(X_train_aspect)}, Test samples: {len(X_test_aspect)}")
        else:
            print(f"- Aspect '{aspek_name}': Skipping split due to insufficient data (only 1 sample).")
            continue

        split_data_per_aspek[aspek_name] = {
            'X_train': X_train_aspect,
            'X_test': X_test_aspect,
            'Y_train': Y_train_sentiment,
            'Y_test': Y_test_sentiment
        }

    print("Data splitting per aspect complete.")

    print("\nVerification of split data shapes (all aspects with splits):")
    for aspek_name, data in split_data_per_aspek.items():
        if 'X_train' in data and 'X_test' in data:
            print(f"\nAspect: {aspek_name}")
            print(f"  X_train shape: {data['X_train'].shape}")
            print(f"  X_test shape: {data['X_test'].shape}")
            print(f"  Y_train shape: {data['Y_train'].shape}")
            print(f"  Y_test shape: {data['Y_test'].shape}")
else:
    print("data_per_aspek is empty or not defined, skipping data splitting.")

Performing train-test split for each aspect...
- Aspect 'fasilitas dan infrastruktur': Data split. Train samples: 722, Test samples: 181
- Aspect 'kualitas pelayanan medis dan staf': Data split. Train samples: 1508, Test samples: 378
- Aspect 'lainnya': Data split. Train samples: 1695, Test samples: 424
- Aspect 'waktu tunggu': Data split. Train samples: 208, Test samples: 53
- Aspect 'biaya layanan': Data split. Train samples: 68, Test samples: 18
Data splitting per aspect complete.

Verification of split data shapes (all aspects with splits):

Aspect: fasilitas dan infrastruktur
  X_train shape: (722,)
  X_test shape: (181,)
  Y_train shape: (722,)
  Y_test shape: (181,)

Aspect: kualitas pelayanan medis dan staf
  X_train shape: (1508,)
  X_test shape: (378,)
  Y_train shape: (1508,)
  Y_test shape: (378,)

Aspect: lainnya
  X_train shape: (1695,)
  X_test shape: (424,)
  Y_train shape: (1695,)
  Y_test shape: (424,)

Aspect: waktu tunggu
  X_train shape: (208,)
  X_test shape: (53,

In [17]:
import pickle
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences
import ast
import pandas as pd # Ensure pandas is imported

# Use SAVE_ROOT_PATH which correctly points to WORK_DIR/EPOCH 10 (folder kerja Kaggle)
# for constructing the path to the experiment_states directory.
# BASE_DRIVE_PATH currently holds a different WORK_DIR subfolder, which is not correct here.
with open(os.path.join(SAVE_ROOT_PATH, 'experiment_states', 'tokenizer_lstm.pkl'), "rb") as f:
    tokenizer = pickle.load(f)
print("Tokenizer loaded successfully.")

maxlen = 20 # Standardized maxlen
print("Sequences dan padding untuk tiap aspek")
for aspek_name, data in split_data_per_aspek.items():
    print(f"Processing aspek: {aspek_name}")

    def prepare_for_tokenizer(series):
        processed_texts = []
        for item in series:
            if isinstance(item, str):
                try:
                    item = ast.literal_eval(item)
                except (ValueError, SyntaxError):
                    item = []
            if isinstance(item, list) and item:
                processed_texts.append(' '.join(item))
            else:
                processed_texts.append('')
        return pd.Series(processed_texts, index=series.index)

    X_train_texts = prepare_for_tokenizer(data['X_train'])
    X_test_texts = prepare_for_tokenizer(data['X_test'])

    X_train_seq = tokenizer.texts_to_sequences(X_train_texts)
    X_test_seq = tokenizer.texts_to_sequences(X_test_texts)

    # Pad sequences
    X_train_padded = pad_sequences(X_train_seq, maxlen=maxlen, padding='post', truncating='post')
    X_test_padded = pad_sequences(X_test_seq, maxlen=maxlen, padding='post', truncating='post')

    data['X_train_padded'] = X_train_padded
    data['X_test_padded'] = X_test_padded

print("Tokenization and padding complete for all aspek.")

print("Verification of padded data shapes (all aspek):")
for aspek_name, data in split_data_per_aspek.items():
    if 'X_train_padded' in data and 'X_test_padded' in data:
        print(f"aspek: {aspek_name}")
        print(f"  X_train_padded shape: {data['X_train_padded'].shape}")
        print(f"  X_test_padded shape: {data['X_test_padded'].shape}")

Tokenizer loaded successfully.
Sequences dan padding untuk tiap aspek
Processing aspek: fasilitas dan infrastruktur
Processing aspek: kualitas pelayanan medis dan staf
Processing aspek: lainnya
Processing aspek: waktu tunggu
Processing aspek: biaya layanan
Tokenization and padding complete for all aspek.
Verification of padded data shapes (all aspek):
aspek: fasilitas dan infrastruktur
  X_train_padded shape: (722, 20)
  X_test_padded shape: (181, 20)
aspek: kualitas pelayanan medis dan staf
  X_train_padded shape: (1508, 20)
  X_test_padded shape: (378, 20)
aspek: lainnya
  X_train_padded shape: (1695, 20)
  X_test_padded shape: (424, 20)
aspek: waktu tunggu
  X_train_padded shape: (208, 20)
  X_test_padded shape: (53, 20)
aspek: biaya layanan
  X_train_padded shape: (68, 20)
  X_test_padded shape: (18, 20)


In [18]:
import pickle
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences
import ast
import pandas as pd # Ensure pandas is imported

# Use SAVE_ROOT_PATH which correctly points to WORK_DIR/EPOCH 05 (folder kerja Kaggle)
# for constructing the path to the experiment_states directory.
# BASE_DRIVE_PATH currently holds a different WORK_DIR subfolder, which is not correct here.
with open(os.path.join(SAVE_ROOT_PATH, 'experiment_states', 'tokenizer_lstm.pkl'), "rb") as f:
    tokenizer = pickle.load(f)
print("Tokenizer loaded successfully.")

maxlen = 20 # Standardized maxlen
print("Sequences dan padding untuk tiap aspek")
for aspek_name, data in split_data_per_aspek.items():
    print(f"Processing aspek: {aspek_name}")

    def prepare_for_tokenizer(series):
        processed_texts = []
        for item in series:
            if isinstance(item, str):
                try:
                    item = ast.literal_eval(item)
                except (ValueError, SyntaxError):
                    item = []
            if isinstance(item, list) and item:
                processed_texts.append(' '.join(item))
            else:
                processed_texts.append('')
        return pd.Series(processed_texts, index=series.index)

    X_train_texts = prepare_for_tokenizer(data['X_train'])
    X_test_texts = prepare_for_tokenizer(data['X_test'])

    X_train_seq = tokenizer.texts_to_sequences(X_train_texts)
    X_test_seq = tokenizer.texts_to_sequences(X_test_texts)

    # Pad sequences
    X_train_padded = pad_sequences(X_train_seq, maxlen=maxlen, padding='post', truncating='post')
    X_test_padded = pad_sequences(X_test_seq, maxlen=maxlen, padding='post', truncating='post')

    data['X_train_padded'] = X_train_padded
    data['X_test_padded'] = X_test_padded

print("Tokenization and padding complete for all aspek.")

print("Verification of padded data shapes (all aspek):")
for aspek_name, data in split_data_per_aspek.items():
    if 'X_train_padded' in data and 'X_test_padded' in data:
        print(f"aspek: {aspek_name}")
        print(f"  X_train_padded shape: {data['X_train_padded'].shape}")
        print(f"  X_test_padded shape: {data['X_test_padded'].shape}")

Tokenizer loaded successfully.
Sequences dan padding untuk tiap aspek
Processing aspek: fasilitas dan infrastruktur
Processing aspek: kualitas pelayanan medis dan staf
Processing aspek: lainnya
Processing aspek: waktu tunggu
Processing aspek: biaya layanan
Tokenization and padding complete for all aspek.
Verification of padded data shapes (all aspek):
aspek: fasilitas dan infrastruktur
  X_train_padded shape: (722, 20)
  X_test_padded shape: (181, 20)
aspek: kualitas pelayanan medis dan staf
  X_train_padded shape: (1508, 20)
  X_test_padded shape: (378, 20)
aspek: lainnya
  X_train_padded shape: (1695, 20)
  X_test_padded shape: (424, 20)
aspek: waktu tunggu
  X_train_padded shape: (208, 20)
  X_test_padded shape: (53, 20)
aspek: biaya layanan
  X_train_padded shape: (68, 20)
  X_test_padded shape: (18, 20)


In [ ]:
import tensorflow as tf
import numpy as np
import random
import os

from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import (
    Dropout,
    Dense,
    LSTM,
    Embedding
)
from tensorflow.keras.models import Sequential
# from tensorflow.keras.callbacks import EarlyStopping # Removed EarlyStopping

from sklearn.model_selection import KFold

# =========================
# RANDOM SEED
# =========================
SEED = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)

# =========================
# PARAMETER
# =========================
vocab_size = 5000
max_len = 20
EPOCHS = 20

# =========================
# PARAMETER UJI COBA DROPOUT
# =========================
if 'DROPOUT_VALUES' not in globals():
    DROPOUT_VALUES = [0.2, 0.5, 0.8]

print(f"Dropout yang akan diuji untuk sentimen per aspek: {DROPOUT_VALUES}")

# =========================
# LOOP TIAP ASPEK
# =========================
for aspek_name, data in split_data_per_aspek.items():

    print("="*70)
    print(f"TRAINING SENTIMEN UNTUK ASPEK: {aspek_name}")
    print("="*70)

    # =========================
    # DATA
    # =========================
    X_pad = data['X_train_padded']
    y = np.array(data['Y_train'])

    num_classes = len(np.unique(y))

    # =========================
    # FOLDER SAVE
    # =========================
    if 'BASE_DRIVE_PATH' not in globals():

        print("❌ BASE_DRIVE_PATH not found.")

        raise SystemExit(
            "Stopping execution: BASE_DRIVE_PATH is not defined."
        )

    save_dir_relative = (
        f"sentiment_models_per_aspect/epoch_{EPOCHS}"
    )

    save_dir_full_path = os.path.join(
        BASE_DRIVE_PATH,
        save_dir_relative
    )

    os.makedirs(
        save_dir_full_path,
        exist_ok=True
    )

    # =========================
    # LOOP TIAP NILAI DROPOUT
    # =========================
    for dropout_rate in DROPOUT_VALUES:

        dropout_tag = str(dropout_rate).replace('.', '_')

        print("\n" + "#"*60)
        print(f"# ASPEK '{aspek_name}' | DROPOUT = {dropout_rate}")
        print("#"*60)

        # =========================
        # KFOLD
        # =========================
        skf = KFold(
            n_splits=5,
            shuffle=True,
            random_state=SEED
        )

        print("="*60)
        print(f"TRAINING DENGAN MAX EPOCH = {EPOCHS}")
        print("="*60)

        # best_val_loss = float('inf') # Removed as EarlyStopping is removed
        fold_no = 1

        fold_accuracies = []

        # =========================
        # KFOLD LOOP
        # =========================
        for train_idx, val_idx in skf.split(X_pad):

            print(f"\n====== Dropout {dropout_rate} | Fold {fold_no} ======")

            # =========================
            # SPLIT DATA
            # =========================
            X_tr = X_pad[train_idx]
            X_val = X_pad[val_idx]

            y_tr = y[train_idx]
            y_val = y[val_idx]

            # =========================
            # MODEL LSTM
            # =========================
            model = Sequential([

                Embedding(
                    input_dim=vocab_size,
                    output_dim=128,
                    mask_zero=True
                ),

                LSTM(
                    128,
                    return_sequences=True,
                    dropout=dropout_rate,
                    recurrent_dropout=dropout_rate
                ),

                LSTM(
                    64,
                    dropout=dropout_rate,
                    recurrent_dropout=dropout_rate
                ),

                Dense(
                    128,
                    activation='relu'
                ),

                Dropout(dropout_rate),

                Dense(
                    num_classes,
                    activation='softmax'
                )
            ])

            # =========================
            # COMPILE
            # =========================
            model.compile(
                optimizer=Adam(
                    learning_rate=0.0005
                ),
                loss='sparse_categorical_crossentropy',
                metrics=['accuracy']
            )

            # =========================
            # EARLY STOPPING - REMOVED
            # =========================
            # early_stopping = EarlyStopping(
            #     monitor='val_loss',
            #     patience=2,
            #     restore_best_weights=True,
            #     verbose=1
            # )

            # =========================
            # TRAINING
            # =========================
            history = model.fit(
                X_tr,
                y_tr,
                epochs=EPOCHS,
                batch_size=32,
                validation_data=(X_val, y_val),
                # callbacks=[early_stopping], # Removed early_stopping from callbacks
                verbose=1
            )

            # =========================
            # EVALUASI
            # =========================
            scores = model.evaluate(
                X_val,
                y_val,
                verbose=0
            )

            print(f"\nDropout {dropout_rate} | Fold {fold_no}")
            print(f"Loss     : {scores[0]:.4f}")
            print(f"Accuracy : {scores[1]:.4f}")

            fold_accuracies.append(
                scores[1]
            )

            # =========================
            # SAVE MODEL FOR EACH FOLD
            # =========================
            # Nama file sekarang menyertakan nilai dropout supaya tiap konfigurasi
            # bisa dibandingkan pada cell evaluasi berikutnya.
            file_name = (
                f"sentiment_model_{aspek_name.replace(' ', '_').replace('/', '_').lower()}"
                f"_dropout_{dropout_tag}_fold_{fold_no}.keras"
            )

            model.save(
                os.path.join(
                    save_dir_full_path,
                    file_name
                )
            )

            print(
                f"Model for '{aspek_name}' Dropout {dropout_rate} Fold {fold_no} saved to:\n"
                f"{os.path.join(save_dir_full_path, file_name)}"
            )

            fold_no += 1

        # =========================
        # RATA-RATA KFOLD UNTUK DROPOUT INI
        # =========================
        avg_acc = np.mean(
            fold_accuracies
        )

        print("\n" + "="*60)
        print(f"RATA-RATA AKURASI ASPEK '{aspek_name}' | DROPOUT {dropout_rate}")
        print("="*60)

        print(
            f"Average Accuracy : "
            f"{avg_acc:.4f}"
        )


Dropout yang akan diuji untuk sentimen per aspek: [0.2, 0.5, 0.8]
TRAINING SENTIMEN UNTUK ASPEK: fasilitas dan infrastruktur

############################################################
# ASPEK 'fasilitas dan infrastruktur' | DROPOUT = 0.2
############################################################
TRAINING DENGAN MAX EPOCH = 20

====== Dropout 0.2 | Fold 1 ======
Epoch 1/20
19/19 ━━━━━━━━━━━━━━━━━━━━ 11s 193ms/step - accuracy: 0.6863 - loss: 1.0642 - val_accuracy: 0.6621 - val_loss: 1.0011
Epoch 2/20
19/19 ━━━━━━━━━━━━━━━━━━━━ 3s 132ms/step - accuracy: 0.6932 - loss: 0.8740 - val_accuracy: 0.6621 - val_loss: 0.8165
Epoch 3/20
19/19 ━━━━━━━━━━━━━━━━━━━━ 3s 136ms/step - accuracy: 0.6950 - loss: 0.6625 - val_accuracy: 0.6690 - val_loss: 0.7040
Epoch 4/20
19/19 ━━━━━━━━━━━━━━━━━━━━ 3s 137ms/step - accuracy: 0.7938 - loss: 0.5406 - val_accuracy: 0.7586 - val_loss: 0.6171
Epoch 5/20
19/19 ━━━━━━━━━━━━━━━━━━━━ 2s 129ms/step - accuracy: 0.8544 - loss: 0.4321 - val_accuracy: 0.7931 - val_los

In [20]:
import os
import re
import numpy as np
import pandas as pd

from tensorflow.keras.models import load_model

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# ============================================================
# CEK VARIABEL YANG DIBUTUHKAN
# ============================================================

if 'BASE_DRIVE_PATH' not in globals():
    print("Error: BASE_DRIVE_PATH tidak ditemukan.")
    raise SystemExit()

if 'sentiment_le' not in globals():
    print("Warning: sentiment_le tidak ditemukan.")
    sentiment_target_names = [str(i) for i in range(3)]
else:
    sentiment_target_names = list(sentiment_le.classes_)

if 'DROPOUT_VALUES' not in globals():
    DROPOUT_VALUES = [0.2, 0.5, 0.8]

# ============================================================
# LOKASI MODEL
# ============================================================

save_dir_relative = f"sentiment_models_per_aspect/epoch_{EPOCHS}"

sentiment_models_base_dir = os.path.join(
    BASE_DRIVE_PATH,
    save_dir_relative
)

print("="*70)
print("EVALUASI ENSEMBLE MODEL SENTIMEN PER ASPEK (per nilai dropout)")
print("="*70)

def get_fold_number(filename):
    match = re.search(r'fold_(\d+)\.keras', filename)
    if match:
        return int(match.group(1))
    return -1

# Menyimpan ringkasan ensemble untuk semua kombinasi aspek x dropout
all_ensemble_rows = []

# ============================================================
# LOOP SETIAP ASPEK
# ============================================================

for aspek_name, data in split_data_per_aspek.items():

    print("\n")
    print("="*70)
    print(f"EVALUASI ASPEK : {aspek_name}")
    print("="*70)

    # --------------------------------------------------------
    # CEK DATA TEST
    # --------------------------------------------------------

    if (
        'X_test_padded' not in data
        or
        'Y_test' not in data
    ):
        print("Data test tidak ditemukan.")
        continue

    X_test_aspect = data['X_test_padded']
    Y_test_aspect = np.array(data['Y_test'])

    if len(Y_test_aspect) == 0:
        print("Data test kosong.")
        continue

    cleaned_aspek_name = aspek_name.replace(' ', '_').replace('/', '_').lower()

    # --------------------------------------------------------
    # LOOP SETIAP NILAI DROPOUT
    # --------------------------------------------------------

    for dropout_rate in DROPOUT_VALUES:

        dropout_tag = str(dropout_rate).replace('.', '_')

        print("\n" + "-"*70)
        print(f"ASPEK '{aspek_name}' | DROPOUT = {dropout_rate}")
        print("-"*70)

        # --------------------------------------------------------
        # CARI MODEL UNTUK ASPEK + DROPOUT INI
        # --------------------------------------------------------

        prefix = f"sentiment_model_{cleaned_aspek_name}_dropout_{dropout_tag}_fold_"

        aspect_model_files = [
            f for f in os.listdir(sentiment_models_base_dir)
            if f.startswith(prefix) and f.endswith('.keras')
        ]

        if not aspect_model_files:
            print(f"Tidak ditemukan model untuk aspek '{aspek_name}' dengan dropout {dropout_rate}")
            continue

        aspect_model_files = sorted(aspect_model_files, key=get_fold_number)

        print(f"Jumlah model ditemukan : {len(aspect_model_files)}")

        # --------------------------------------------------------
        # LOAD SEMUA MODEL (5-fold) UNTUK DROPOUT INI
        # --------------------------------------------------------

        all_y_pred_probs = []

        for model_file in aspect_model_files:

            full_model_path = os.path.join(sentiment_models_base_dir, model_file)

            try:
                model = load_model(full_model_path)
                y_pred_prob = model.predict(X_test_aspect, verbose=0)
                all_y_pred_probs.append(y_pred_prob)
            except Exception as e:
                print(f"Error model {model_file}: {e}")

        if len(all_y_pred_probs) == 0:
            print("Tidak ada prediksi yang berhasil.")
            continue

        # --------------------------------------------------------
        # ENSEMBLE SOFT VOTING (antar 5 fold, untuk dropout ini)
        # --------------------------------------------------------

        avg_y_pred_prob = np.mean(all_y_pred_probs, axis=0)
        y_pred_ensemble = np.argmax(avg_y_pred_prob, axis=1)

        # --------------------------------------------------------
        # METRICS
        # --------------------------------------------------------

        accuracy = accuracy_score(Y_test_aspect, y_pred_ensemble)
        precision = precision_score(Y_test_aspect, y_pred_ensemble, average='weighted', zero_division=0)
        recall = recall_score(Y_test_aspect, y_pred_ensemble, average='weighted', zero_division=0)
        f1 = f1_score(Y_test_aspect, y_pred_ensemble, average='weighted', zero_division=0)

        all_ensemble_rows.append({
            'aspek': aspek_name,
            'dropout': dropout_rate,
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1_score': f1
        })

        print("\n" + "-"*60)
        print("HASIL EVALUASI ENSEMBLE")
        print("-"*60)

        print(f"Accuracy      : {accuracy:.4f}")
        print(f"Precision     : {precision:.4f}")
        print(f"Recall        : {recall:.4f}")
        print(f"F1-Score      : {f1:.4f}")

        print("\nClassification Report:")
        print(
            classification_report(
                Y_test_aspect,
                y_pred_ensemble,
                target_names=sentiment_target_names,
                zero_division=0
            )
        )

        print("\nConfusion Matrix:")
        print(confusion_matrix(Y_test_aspect, y_pred_ensemble))

# ============================================================
# RINGKASAN SEMUA ASPEK x DROPOUT
# ============================================================

if all_ensemble_rows:
    df_sentiment_dropout_summary = pd.DataFrame(all_ensemble_rows)

    print("\n" + "#"*70)
    print("# RINGKASAN ENSEMBLE SENTIMEN PER ASPEK x DROPOUT")
    print("#"*70)
    display(df_sentiment_dropout_summary.sort_values(by=['aspek', 'accuracy'], ascending=[True, False]))

    print("\n" + "#"*70)
    print("# RATA-RATA AKURASI PER DROPOUT (semua aspek digabung)")
    print("#"*70)
    display(
        df_sentiment_dropout_summary
        .groupby('dropout')[['accuracy', 'precision', 'recall', 'f1_score']]
        .mean()
        .sort_values(by='accuracy', ascending=False)
    )

# ============================================================
# SELESAI
# ============================================================

print("\n")
print("="*70)
print("EVALUASI MODEL SENTIMEN PER ASPEK SELESAI")
print("="*70)



------------------------------------------------------------
HASIL EVALUASI ENSEMBLE
------------------------------------------------------------
Accuracy      : 0.9180
Precision     : 0.9176
Recall        : 0.9180
F1-Score      : 0.9177

Classification Report:
              precision    recall  f1-score   support

    Negative       0.33      0.33      0.33         9
     Neutral       0.90      0.88      0.89       124
    Positive       0.95      0.96      0.95       245

    accuracy                           0.92       378
   macro avg       0.73      0.72      0.73       378
weighted avg       0.92      0.92      0.92       378


Confusion Matrix:
[[  3   3   3]
 [  5 109  10]
 [  1   9 235]]

----------------------------------------------------------------------
ASPEK 'kualitas pelayanan medis dan staf' | DROPOUT = 0.8
----------------------------------------------------------------------
Jumlah model ditemukan : 5

------------------------------------------------------------
H

,aspek,dropout,accuracy,precision,recall,f1_score
12,biaya layanan,0.2,0.555556,0.308642,0.555556,0.396825
13,biaya layanan,0.5,0.555556,0.308642,0.555556,0.396825
14,biaya layanan,0.8,0.555556,0.308642,0.555556,0.396825
1,fasilitas dan infrastruktur,0.5,0.933702,0.933137,0.933702,0.933327
0,fasilitas dan infrastruktur,0.2,0.928177,0.928702,0.928177,0.927480
2,fasilitas dan infrastruktur,0.8,0.867403,0.808804,0.867403,0.834432
5,kualitas pelayanan medis dan staf,0.8,0.933862,0.925162,0.933862,0.927960
3,kualitas pelayanan medis dan staf,0.2,0.923280,0.928303,0.923280,0.924736
4,kualitas pelayanan medis dan staf,0.5,0.917989,0.917618,0.917989,0.917737
7,lainnya,0.5,0.830189,0.829220,0.830189,0.829531



######################################################################
# RATA-RATA AKURASI PER DROPOUT (semua aspek digabung)
######################################################################


,accuracy,precision,recall,f1_score
dropout,,,,
0.5,0.813525,0.754905,0.813525,0.772063
0.2,0.807818,0.746124,0.807818,0.768467
0.8,0.788345,0.689348,0.788345,0.727394




EVALUASI MODEL SENTIMEN PER ASPEK SELESAI


In [21]:
import os
import re
import numpy as np
import pandas as pd
from tensorflow.keras.models import load_model
from sklearn.metrics import accuracy_score

# Define the directory to save the best sentiment models
best_sentiment_models_dir = os.path.join(BASE_DRIVE_PATH, 'best_sentiment_models')
os.makedirs(best_sentiment_models_dir, exist_ok=True)

# Path where individual fold models for each aspect are saved
# Assuming EPOCHS is still defined and correct for the folder name
if 'EPOCHS' not in globals():
    EPOCHS = 20 # Fallback if EPOCHS is not defined
sentiment_models_base_dir = os.path.join(BASE_DRIVE_PATH, f"sentiment_models_per_aspect/epoch_{EPOCHS}")

print("Menyimpan model terbaik untuk setiap klasifikasi sentimen per aspek (lintas semua nilai dropout)...")

def parse_dropout_fold(filename):
    match = re.search(r'dropout_([0-9_]+)_fold_(\d+)\.keras', filename)
    if match:
        return float(match.group(1).replace('_', '.')), int(match.group(2))
    return None, -1

# Iterate through each aspect's split data
if 'split_data_per_aspek' in globals() and split_data_per_aspek:
    for aspek_name, data in split_data_per_aspek.items():
        print(f"\n--- Memproses aspek: {aspek_name} ---")

        # Check if test data exists for this aspect
        if 'X_test_padded' not in data or 'Y_test' not in data:
            print(f"  Tidak ada data uji yang diproses untuk aspek '{aspek_name}'. Melewatkan.")
            continue
        if len(data['Y_test']) == 0:
            print(f"  Data uji kosong untuk aspek '{aspek_name}'. Melewatkan.")
            continue

        X_test_aspect = data['X_test_padded']
        Y_test_aspect = np.array(data['Y_test'])

        # Clean aspect_name for filename matching (replace spaces/slashes with underscores and lowercase)
        cleaned_aspek_name = aspek_name.replace(' ', '_').replace('/', '_').lower()

        # Filter model files for the current aspect (semua nilai dropout & fold)
        aspect_model_files = [
            f for f in os.listdir(sentiment_models_base_dir)
            if f.startswith(f"sentiment_model_{cleaned_aspek_name}_dropout_") and f.endswith('.keras')
        ]

        if not aspect_model_files:
            print(f"  Tidak ditemukan model untuk aspek '{aspek_name}' di '{sentiment_models_base_dir}'.")
            continue

        best_accuracy_for_aspect = -1
        best_model_filename = ""
        best_dropout_for_aspect = None
        best_fold_for_aspect = None

        # Evaluate each model (semua dropout & fold) for the current aspect to find the best one
        for model_file in aspect_model_files:
            full_model_path = os.path.join(sentiment_models_base_dir, model_file)
            try:
                model = load_model(full_model_path)

                # Predict probabilities
                y_pred_prob = model.predict(X_test_aspect, verbose=0)
                # Get predicted classes
                y_pred = np.argmax(y_pred_prob, axis=1)

                # Calculate accuracy
                current_accuracy = accuracy_score(Y_test_aspect, y_pred)

                if current_accuracy > best_accuracy_for_aspect:
                    best_accuracy_for_aspect = current_accuracy
                    best_model_filename = model_file
                    best_dropout_for_aspect, best_fold_for_aspect = parse_dropout_fold(model_file)

            except Exception as e:
                print(f"  Error saat mengevaluasi model '{model_file}': {e}")
                continue

        if best_model_filename:
            source_best_model_path = os.path.join(sentiment_models_base_dir, best_model_filename)
            destination_best_model_path = os.path.join(best_sentiment_models_dir, f"{cleaned_aspek_name}_best_model.keras")

            try:
                # Load the best model again to save it in the new location
                best_model = load_model(source_best_model_path)
                best_model.save(destination_best_model_path)
                print(
                    f"  ✅ Model terbaik untuk aspek '{aspek_name}' "
                    f"(Dropout: {best_dropout_for_aspect}, Fold: {best_fold_for_aspect}, "
                    f"Akurasi: {best_accuracy_for_aspect:.4f}) disimpan sebagai: '{destination_best_model_path}'"
                )
            except Exception as e:
                print(f"  ❌ Error saat menyimpan model terbaik untuk aspek '{aspek_name}': {e}")
        else:
            print(f"  Tidak ada model terbaik yang ditemukan untuk aspek '{aspek_name}'.")

else:
    print("❌ Variabel 'split_data_per_aspek' tidak ditemukan atau kosong. Pastikan data telah dibagi per aspek.")

print("\nProses penyimpanan model terbaik per aspek selesai.")


Menyimpan model terbaik untuk setiap klasifikasi sentimen per aspek (lintas semua nilai dropout)...

--- Memproses aspek: fasilitas dan infrastruktur ---
  ✅ Model terbaik untuk aspek 'fasilitas dan infrastruktur' (Dropout: 0.5, Fold: 1, Akurasi: 0.9392) disimpan sebagai: '/kaggle/working/EPOCH_20/best_sentiment_models/fasilitas_dan_infrastruktur_best_model.keras'

--- Memproses aspek: kualitas pelayanan medis dan staf ---
  ✅ Model terbaik untuk aspek 'kualitas pelayanan medis dan staf' (Dropout: 0.8, Fold: 2, Akurasi: 0.9471) disimpan sebagai: '/kaggle/working/EPOCH_20/best_sentiment_models/kualitas_pelayanan_medis_dan_staf_best_model.keras'

--- Memproses aspek: lainnya ---
  ✅ Model terbaik untuk aspek 'lainnya' (Dropout: 0.8, Fold: 1, Akurasi: 0.8325) disimpan sebagai: '/kaggle/working/EPOCH_20/best_sentiment_models/lainnya_best_model.keras'

--- Memproses aspek: waktu tunggu ---
  ✅ Model terbaik untuk aspek 'waktu tunggu' (Dropout: 0.2, Fold: 5, Akurasi: 0.8491) disimpan sebagai